In [0]:
import json, requests
from datetime import datetime, timezone

FEED = "https://cdn.mbta.com/realtime/VehiclePositions_enhanced.json"
BASE = "/Volumes/transit/bronze/landing/rt/vehicle_positions"

resp = requests.get(FEED, timeout=30)
resp.raise_for_status()
raw = resp.content

# Filename is deterministic on the feed's own header timestamp, so a
# duplicate fetch overwrites rather than adds a near-identical snapshot.
header_ts = json.loads(raw)["header"]["timestamp"]
dt = datetime.fromtimestamp(header_ts, tz=timezone.utc).strftime("%Y-%m-%d")

out_dir = f"{BASE}/dt={dt}"
dbutils.fs.mkdirs(out_dir)
out_path = f"{out_dir}/snapshot_{header_ts}.json"

with open(out_path, "wb") as f:
    f.write(raw)          # verbatim bytes, no parsing

print(f"{out_path}  {len(raw):,} bytes  {len(json.loads(raw)['entity'])} entities")